<a href="https://colab.research.google.com/github/jorgemunozl/vla-test/blob/main/sixth_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast In Slow First Tests

In [1]:
import os

# Test

# Set this BEFORE importing pytorch/tensorflow
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
# Check if it worked (should return 1 if you selected a single GPU)
print(torch.cuda.device_count()) 

1


In [2]:
# DS_ID = "NONHUMAN-RESEARCH/general-task-index"
DS_ID = "NONHUMAN-RESEARCH/pick-and-place-all-fruits-annotated"
PRETRAINED_PATH = "lerobot/pi05_base"

In [3]:
from xhuman.policies.fis.configuration_fis import FISConfig

policy_config = FISConfig(repo_id="none",device="cuda",pretrained_path=PRETRAINED_PATH)

/home/lperez/miniconda3/envs/xhuman_nh/lib/python3.12/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.3 is installed, but it is not compatible with the installed jaxlib version 0.7.2, so it will not be used.
  warnings.warn(


In [4]:
from xhuman.configs.default import LerobotDatasetConfig

dataset_config = LerobotDatasetConfig(
    repo_id=DS_ID,
)

In [5]:
from xhuman.configs.train import TrainPipelineConfigXHUMAN

cfg = TrainPipelineConfigXHUMAN(
    dataset=dataset_config,
    policy=policy_config,
)
cfg.validate()

In [6]:
from xhuman.datasets.factory import make_dataset_xhuman

dataset = make_dataset_xhuman(cfg)

In [7]:
from xhuman.policies.factory import make_xhuman_policy

policy = make_xhuman_policy(
    cfg=cfg.policy,
    ds_meta=dataset.meta,
)

Loading model from: lerobot/pi05_base
✓ Loaded state dict from model.safetensors
Remapped: action_in_proj.bias -> model.action_in_proj.bias
Remapped: action_in_proj.weight -> model.action_in_proj.weight
Remapped: action_out_proj.bias -> model.action_out_proj.bias
Remapped: action_out_proj.weight -> model.action_out_proj.weight
Remapped: paligemma_with_expert.gemma_expert.lm_head.weight -> model.paligemma_with_expert.gemma_expert.lm_head.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight
Remapped: paligemma_with_expert.gemma_exp

In [8]:
from xhuman.policies.factory import make_xhuman_pre_post_processors

preprocessor, _ = make_xhuman_pre_post_processors(
        policy_cfg=policy_config,
        dataset_stats=dataset.meta.stats,
    )

In [9]:
device = torch.device("cuda")

In [10]:
frame = dataset[0]
frame.keys()

dict_keys(['observation.images.left', 'observation.images.top', 'observation.images.right', 'action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'general_task_index', 'action_is_pad', 'task', 'general_task'])

In [11]:
batch_slow = preprocessor.slow(frame)

In [12]:
batch_fast = preprocessor.fast(frame)

In [13]:
batch_fast["observation.state"]

tensor([[[-8.0769e+02, -1.0988e+07, -1.0000e+00,  3.1662e+00,  4.5636e-01,
          -3.5195e+00, -1.7731e+01,  9.4314e-01, -9.9952e-01,  9.9899e-01,
           1.9229e-01, -7.0983e-01,  1.0242e+00, -1.0009e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]], device='cuda:0')

In [14]:
batch_slow.keys()

dict_keys(['action', 'next.reward', 'next.done', 'next.truncated', 'info', 'action_is_pad', 'task', 'index', 'task_index', 'episode_index', 'observation.images.left', 'observation.images.top', 'observation.images.right', 'observation.state', 'observation.language.tokens', 'observation.language.attention_mask'])

In [15]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")

In [16]:
tokens_pt = batch_slow["observation.language.tokens"]
words = tokenizer.batch_decode(tokens_pt, skip_special_tokens=True)
words

['pick up the grape and put it in the basket']

In [17]:
prefix_out , kv_cache , pad_mask  = policy.sample_embedding(batch_slow)

In [18]:
prefix_out[0].shape

torch.Size([200, 2048])

In [19]:
batch_slow.keys()

dict_keys(['action', 'next.reward', 'next.done', 'next.truncated', 'info', 'action_is_pad', 'task', 'index', 'task_index', 'episode_index', 'observation.images.left', 'observation.images.top', 'observation.images.right', 'observation.state', 'observation.language.tokens', 'observation.language.attention_mask'])

In [23]:
img_fast, img_masks_fast = policy._preprocess_images(batch_fast)
raw_state = batch_fast["observation.state"]
raw_state

tensor([[[-8.0769e+02, -1.0988e+07, -1.0000e+00,  3.1662e+00,  4.5636e-01,
          -3.5195e+00, -1.7731e+01,  9.4314e-01, -9.9952e-01,  9.9899e-01,
           1.9229e-01, -7.0983e-01,  1.0242e+00, -1.0009e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]], device='cuda:0')

In [21]:
batch_slow["observation.state"]

In [ ]:
policy.model.config.output_features["action"].shape

{'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(14,))}

In [ ]:
actions = policy.model.sample_actions(
    images=img_fast,
    img_masks=img_masks_fast,
    tokens=tokens_slow,
    masks=masks_slow,
)
actions.shape

In [25]:
actions = policy.model.fast_model(
    images_fast=img_fast,
    img_masks_fast=img_masks_fast,
    past_key_values=kv_cache,
    state_embs=raw_state,
    latent_embeds=prefix_out[0],
    prefix_pad_masks=pad_mask,
)
actions.shape

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.cuda.FloatTensor instead (while checking arguments for embedding)